# Phase 4 - Notebook 00: Feed-forward Gaussian Splatting Overview

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase4/00_phase4_overview.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand the fundamental paradigm shift from optimization-based to feed-forward 3DGS
2. Know the key methods: MVSplat, pixelSplat, DepthSplat
3. Understand pixel-aligned Gaussian representation at a high level
4. See the connection between feed-forward 3DGS and SLAM/MVS
5. Have a clear roadmap for Phase 4

**Estimated Time**: 45 minutes

**Prerequisites**: Phase 1 (Notebooks 00-10), basic understanding of multi-view geometry

---

## 0. Environment Setup

In [ ]:
# Environment setup
import os
import sys

# Colab compatibility
if 'COLAB_GPU' in os.environ:
    !pip install -q plotly ipywidgets
    !git clone https://github.com/ChunLI-666/3DGS-from-scratch.git
    %cd 3DGS-from-scratch

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Ellipse
import warnings
warnings.filterwarnings('ignore')

print("Environment ready!")
print(f"NumPy version: {np.__version__}")

## 1. Why Feed-forward Gaussian Splatting?

### Recap: Phase 1 Optimization-based 3DGS

In Phase 1, we learned the original 3DGS pipeline:

```
Multi-view Images + COLMAP Poses
        │
        ▼
  SfM Point Cloud (sparse)
        │
        ▼
  Initialize Gaussians
        │
        ▼
  Per-scene Optimization (30 min+)
  ├── Render from random views
  ├── Compare with ground truth
  ├── Backprop gradients
  └── Adaptive density control
        │
        ▼
  Optimized Gaussian Scene
```

This works great but has **critical limitations**:

| Limitation | Impact |
|------------|--------|
| Per-scene optimization | 30 min+ per scene, no reuse |
| Requires COLMAP | Fails on textureless/repetitive scenes |
| No generalization | New scene = start from scratch |
| Not real-time | Cannot be used in AR/VR/robotics |

In [ ]:
# Visualize the two paradigms side by side

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Left: Optimization-based 3DGS ---
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.set_title('Optimization-based 3DGS (Phase 1)', fontsize=14, fontweight='bold')
ax.axis('off')

# Pipeline boxes
boxes_left = [
    (1, 10.5, 'Multi-view Images\n+ COLMAP Poses', '#E8F5E9'),
    (1, 8.5, 'SfM Point Cloud\n(Initialization)', '#E3F2FD'),
    (1, 6.0, 'Per-scene Optimization\n(30 min - hours)', '#FFF3E0'),
    (1, 3.5, 'Gradient Descent\n+ Density Control', '#FFF3E0'),
    (1, 1.5, 'Gaussian Scene\n(This scene only)', '#FFEBEE'),
]

for (x, y, text, color) in boxes_left:
    box = FancyBboxPatch((x, y-0.6), 8, 1.2, 
                         boxstyle="round,pad=0.1", 
                         facecolor=color, edgecolor='gray', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + 4, y, text, ha='center', va='center', fontsize=10)

# Arrows
for i in range(len(boxes_left)-1):
    y_start = boxes_left[i][1] - 0.6
    y_end = boxes_left[i+1][1] + 0.6
    ax.annotate('', xy=(5, y_end), xytext=(5, y_start),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))

# Loop arrow for optimization
ax.annotate('', xy=(9.2, 6.6), xytext=(9.2, 3.5),
            arrowprops=dict(arrowstyle='->', color='#FF9800', lw=2, 
                           connectionstyle='arc3,rad=-0.3'))
ax.text(9.8, 5.0, 'iterate\n~30k', fontsize=8, color='#FF9800', 
        ha='center', style='italic')

# Timing
ax.text(5, 0.3, 'Per scene: 30 min - hours', fontsize=11, 
        ha='center', color='#D32F2F', fontweight='bold')


# --- Right: Feed-forward 3DGS ---
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.set_title('Feed-forward 3DGS (Phase 4)', fontsize=14, fontweight='bold')
ax.axis('off')

# Two stages
# Stage 1: Pre-training (done once)
ax.text(5, 11.5, 'Stage 1: Pre-training (done once)', fontsize=11, 
        ha='center', color='#1565C0', fontweight='bold')

boxes_train = [
    (1, 10.2, 'Large-scale Dataset\n(RE10K, ACID, etc.)', '#E3F2FD'),
    (1, 8.2, 'Train Neural Network\n(days on GPU cluster)', '#E3F2FD'),
]

for (x, y, text, color) in boxes_train:
    box = FancyBboxPatch((x, y-0.6), 8, 1.2,
                         boxstyle="round,pad=0.1",
                         facecolor=color, edgecolor='#1565C0', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + 4, y, text, ha='center', va='center', fontsize=10)

ax.annotate('', xy=(5, 8.8), xytext=(5, 9.6),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))

# Separator
ax.plot([0.5, 9.5], [7.2, 7.2], '--', color='gray', alpha=0.5)

# Stage 2: Inference (per scene)
ax.text(5, 6.8, 'Stage 2: Inference (per new scene)', fontsize=11,
        ha='center', color='#2E7D32', fontweight='bold')

boxes_infer = [
    (1, 5.5, '2-5 Input Images\n(no poses needed)', '#E8F5E9'),
    (1, 3.5, 'Neural Network\nForward Pass', '#E8F5E9'),
    (1, 1.5, 'Gaussian Scene\n(any scene!)', '#E8F5E9'),
]

for (x, y, text, color) in boxes_infer:
    box = FancyBboxPatch((x, y-0.6), 8, 1.2,
                         boxstyle="round,pad=0.1",
                         facecolor=color, edgecolor='#2E7D32', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + 4, y, text, ha='center', va='center', fontsize=10)

for i in range(len(boxes_infer)-1):
    y_start = boxes_infer[i][1] - 0.6
    y_end = boxes_infer[i+1][1] + 0.6
    ax.annotate('', xy=(5, y_end), xytext=(5, y_start),
                arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))

# Timing
ax.text(5, 0.3, 'Per scene: < 1 second', fontsize=11,
        ha='center', color='#2E7D32', fontweight='bold')

plt.tight_layout()
plt.savefig('paradigm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Left: Original 3DGS requires per-scene optimization (slow, no generalization)")
print("Right: Feed-forward 3DGS uses a pretrained network (fast, generalizes)")

## 2. The Feed-forward Paradigm

### Core Idea

Instead of optimizing Gaussian parameters per scene, **train a neural network** to predict them:

```
Input: 2-5 images of a new scene
       │
       ▼
  Neural Network (pretrained)
  ├── Feature Extraction
  ├── Cross-view Reasoning  
  └── Gaussian Prediction
       │
       ▼
Output: Complete set of 3D Gaussians
        (positions, covariances, opacities, colors)
        │
        ▼
  Standard 3DGS Rendering
  (same rasterizer as Phase 1)
```

### Key Insight: Pixel-aligned Gaussians

Feed-forward methods predict **one Gaussian per pixel**:
- For each pixel in each input view, predict a 3D Gaussian
- Gaussian center = back-project pixel to 3D using predicted depth
- Gaussian properties (scale, rotation, opacity) = network output
- Color = image pixel color (or predicted via SH)

This is fundamentally different from original 3DGS where Gaussians are freely positioned.

In [ ]:
# Visualize pixel-aligned vs free-form Gaussians

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

np.random.seed(42)

# --- Panel 1: Original 3DGS (free-form) ---
ax = axes[0]
ax.set_title('Original 3DGS\n(Free-form Gaussians)', fontsize=12, fontweight='bold')

# Random Gaussian positions (after optimization)
n_gaussians = 80
positions = np.random.randn(n_gaussians, 2) * 1.5
# Cluster them to simulate scene structure
cluster_centers = [(-1.5, -1), (1.5, 0.5), (0, 1.5), (-0.5, -2)]
for i, (cx, cy) in enumerate(cluster_centers):
    n = n_gaussians // len(cluster_centers)
    start = i * n
    positions[start:start+n, 0] = np.random.randn(n) * 0.6 + cx
    positions[start:start+n, 1] = np.random.randn(n) * 0.6 + cy

for i in range(n_gaussians):
    sx = np.random.rand() * 0.4 + 0.1
    sy = np.random.rand() * 0.4 + 0.1
    angle = np.random.rand() * 360
    color = plt.cm.Set2(np.random.rand())
    ellipse = Ellipse(positions[i], sx, sy, angle=angle,
                     fill=True, alpha=0.35, facecolor=color, edgecolor='gray', lw=0.5)
    ax.add_patch(ellipse)
    ax.plot(positions[i, 0], positions[i, 1], 'k.', markersize=2)

ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_aspect('equal')
ax.text(0, -3.7, 'Irregular spacing, varying sizes', ha='center', fontsize=9, style='italic')

# --- Panel 2: Image grid ---
ax = axes[1]
ax.set_title('Input Image\n(Pixel Grid)', fontsize=12, fontweight='bold')

# Draw pixel grid
grid_size = 12
for i in range(grid_size + 1):
    ax.plot([i - grid_size/2, i - grid_size/2], [-grid_size/2, grid_size/2], 
            'k-', alpha=0.2, lw=0.5)
    ax.plot([-grid_size/2, grid_size/2], [i - grid_size/2, i - grid_size/2], 
            'k-', alpha=0.2, lw=0.5)

# Color some pixels to simulate an image
for i in range(grid_size):
    for j in range(grid_size):
        x = i - grid_size/2 + 0.5
        y = j - grid_size/2 + 0.5
        r = np.sqrt(x**2 + y**2)
        color_val = np.clip(1 - r/8, 0.1, 1.0)
        rect = plt.Rectangle((x - 0.5, y - 0.5), 1, 1, 
                             facecolor=plt.cm.viridis(color_val), alpha=0.7)
        ax.add_patch(rect)

ax.set_xlim(-grid_size/2 - 0.5, grid_size/2 + 0.5)
ax.set_ylim(-grid_size/2 - 0.5, grid_size/2 + 0.5)
ax.set_aspect('equal')
ax.text(0, -grid_size/2 - 1.2, 'Each pixel predicts one Gaussian', 
        ha='center', fontsize=9, style='italic')

# Arrow from panel 2 to panel 3
fig.patches.append(FancyArrowPatch(
    (0.63, 0.5), (0.68, 0.5),
    transform=fig.transFigure,
    arrowstyle='->', mutation_scale=20,
    color='#2E7D32', lw=3
))

# --- Panel 3: Pixel-aligned Gaussians ---
ax = axes[2]
ax.set_title('Feed-forward 3DGS\n(Pixel-aligned Gaussians)', fontsize=12, fontweight='bold')

# Regular grid of Gaussians
for i in range(grid_size):
    for j in range(grid_size):
        x = i - grid_size/2 + 0.5
        y = j - grid_size/2 + 0.5
        # Add small jitter to simulate depth variation
        jx = x + np.random.randn() * 0.05
        jy = y + np.random.randn() * 0.05
        r = np.sqrt(x**2 + y**2)
        # Scale varies by "depth"
        sx = 0.3 + r * 0.03
        sy = 0.3 + r * 0.03
        angle = np.random.rand() * 30
        color_val = np.clip(1 - r/8, 0.1, 1.0)
        ellipse = Ellipse((jx, jy), sx, sy, angle=angle,
                         fill=True, alpha=0.35, 
                         facecolor=plt.cm.viridis(color_val),
                         edgecolor='gray', lw=0.3)
        ax.add_patch(ellipse)

ax.set_xlim(-grid_size/2 - 0.5, grid_size/2 + 0.5)
ax.set_ylim(-grid_size/2 - 0.5, grid_size/2 + 0.5)
ax.set_aspect('equal')
ax.text(0, -grid_size/2 - 1.2, 'Regular grid, structured layout', 
        ha='center', fontsize=9, style='italic')

plt.tight_layout()
plt.savefig('pixel_aligned_gaussians.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key difference:")
print("  - Original 3DGS: Gaussians are freely positioned (random init + optimization)")
print("  - Feed-forward: One Gaussian per pixel, structured grid, predicted by network")

## 3. Detailed Paradigm Comparison

Let's systematically compare the two paradigms across every dimension.

In [ ]:
# Comprehensive comparison table

comparison = {
    'Dimension': [
        'Training',
        'Inference time',
        'Camera poses',
        'Generalization',
        'Rendering quality',
        'Input requirement',
        'Gaussian init',
        'Gaussian layout',
        'Densification',
        'Use case',
    ],
    'Optimization (Phase 1)': [
        'Per-scene (30 min+)',
        'N/A (training IS inference)',
        'Required (COLMAP)',
        'None (single scene)',
        'High (fully optimized)',
        '50-300 images per scene',
        'From SfM point cloud',
        'Free-form (adaptive)',
        'Clone / Split / Prune',
        'Offline reconstruction',
    ],
    'Feed-forward (Phase 4)': [
        'Once on large dataset',
        '< 1 second',
        'Optional (can be predicted)',
        'Cross-scene (zero-shot)',
        'Medium-High (improving)',
        '2-5 images per scene',
        'From network prediction',
        'Pixel-aligned (structured)',
        'Not needed',
        'Real-time / AR / VR',
    ],
}

print("=" * 85)
print(f"{'Dimension':25s} | {'Optimization (Phase 1)':28s} | {'Feed-forward (Phase 4)':28s}")
print("=" * 85)
for i in range(len(comparison['Dimension'])):
    d = comparison['Dimension'][i]
    o = comparison['Optimization (Phase 1)'][i]
    f = comparison['Feed-forward (Phase 4)'][i]
    print(f"{d:25s} | {o:28s} | {f:28s}")
print("=" * 85)

In [ ]:
# Visualize quality vs speed trade-off

fig, ax = plt.subplots(figsize=(10, 7))

methods = {
    'Original\n3DGS':       {'speed': 0.0005, 'psnr': 28.5, 'color': '#D32F2F', 'marker': 's'},  # ~30min
    'Mip-\nSplatting':      {'speed': 0.0004, 'psnr': 29.0, 'color': '#C62828', 'marker': 's'},
    'MVSplat':              {'speed': 22,     'psnr': 25.8, 'color': '#2E7D32', 'marker': 'o'},
    'pixelSplat':           {'speed': 10,     'psnr': 24.5, 'color': '#1565C0', 'marker': 'o'},
    'DepthSplat':           {'speed': 15,     'psnr': 26.5, 'color': '#6A1B9A', 'marker': '^'},
    'MVSplat\n(5-view)':    {'speed': 8,      'psnr': 27.2, 'color': '#2E7D32', 'marker': 'D'},
}

for name, data in methods.items():
    ax.scatter(data['speed'], data['psnr'], 
              s=200, c=data['color'], marker=data['marker'],
              edgecolors='black', linewidths=1, zorder=5)
    # Offset text slightly
    offset_x = 0.15 if data['speed'] < 1 else data['speed'] * 0.15
    ax.annotate(name, (data['speed'], data['psnr']),
               textcoords='offset points', xytext=(15, 8),
               fontsize=9, fontweight='bold',
               arrowprops=dict(arrowstyle='-', color='gray', alpha=0.5))

# Regions
ax.axvspan(0.0001, 0.01, alpha=0.08, color='red', label='Optimization-based')
ax.axvspan(1, 100, alpha=0.08, color='green', label='Feed-forward')

ax.set_xscale('log')
ax.set_xlabel('Inference Speed (FPS) [log scale]', fontsize=12)
ax.set_ylabel('PSNR (dB) on RE10K', fontsize=12)
ax.set_title('Quality vs Speed: Optimization vs Feed-forward 3DGS', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.0001, 100)
ax.set_ylim(23, 30)

# Add explanatory text
ax.text(0.001, 23.5, 'Minutes per scene', fontsize=9, color='#D32F2F', ha='center')
ax.text(15, 23.5, 'Real-time capable', fontsize=9, color='#2E7D32', ha='center')

plt.tight_layout()
plt.savefig('quality_vs_speed.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key observation: Feed-forward methods sacrifice some quality for 10000x speed improvement")

## 4. Method Landscape

The feed-forward 3DGS field has evolved rapidly. Here are the key methods:

### 4.1 MVSplat (ECCV 2024)

**Key idea**: Use Multi-View Stereo's **Cost Volume** to reason about geometry

```
2+ Images  ──►  Feature Extraction (U-Net)
                      │
                      ▼
              Cost Volume (Plane Sweeping)
                      │
                      ▼
              3D U-Net Processing
                      │
                      ▼
              Gaussian Prediction (per pixel)
              ├── Depth → 3D position
              ├── Scale, rotation
              ├── Opacity  
              └── Color
```

- **Strength**: Explicit geometry reasoning via Cost Volume, fast (22 FPS)
- **Weakness**: Requires camera poses at inference

### 4.2 pixelSplat (CVPR 2024)

**Key idea**: Learn geometry **implicitly** via cross-attention, no Cost Volume

```
2 Images  ──►  CNN Backbone (EfficientNet)
                      │
                      ▼
              Epipolar Cross-Attention
                      │
                      ▼
              Gaussian Prediction (per pixel)
```

- **Strength**: Elegant end-to-end design
- **Weakness**: Slower (10 FPS), less robust on textureless regions

### 4.3 DepthSplat (2025)

**Key idea**: Integrate monocular depth priors (Depth Anything) with Cost Volume

- Combines the best of learned depth and multi-view geometry
- Better generalization to unseen scenes

In [ ]:
# Method taxonomy visualization

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Feed-forward 3DGS Method Taxonomy', fontsize=16, fontweight='bold')

# Root
root_box = FancyBboxPatch((4.5, 8.5), 5, 1,
                          boxstyle="round,pad=0.15",
                          facecolor='#E8EAF6', edgecolor='#283593', linewidth=2)
ax.add_patch(root_box)
ax.text(7, 9, 'Feed-forward 3DGS', ha='center', va='center',
        fontsize=13, fontweight='bold', color='#283593')

# Branch 1: With Cost Volume
branch1_box = FancyBboxPatch((0.5, 5.5), 5.5, 1.2,
                             boxstyle="round,pad=0.15",
                             facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(branch1_box)
ax.text(3.25, 6.1, 'With Cost Volume\n(Explicit Geometry)', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#2E7D32')

# Branch 2: Without Cost Volume
branch2_box = FancyBboxPatch((7.5, 5.5), 5.5, 1.2,
                             boxstyle="round,pad=0.15",
                             facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(branch2_box)
ax.text(10.25, 6.1, 'Without Cost Volume\n(Implicit Geometry)', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#1565C0')

# Arrows from root to branches
ax.annotate('', xy=(3.25, 6.7), xytext=(5.5, 8.5),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax.annotate('', xy=(10.25, 6.7), xytext=(8.5, 8.5),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))

# Methods under Branch 1
methods_cv = [
    (0.5, 3.5, 'MVSplat\n(ECCV 2024)', '#C8E6C9', '22 FPS\nPSNR ~25.8'),
    (0.5, 1.2, 'DepthSplat\n(2025)', '#C8E6C9', '15 FPS\nPSNR ~26.5'),
]

for (x, y, name, color, info) in methods_cv:
    box = FancyBboxPatch((x, y), 3.5, 1.2,
                         boxstyle="round,pad=0.1",
                         facecolor=color, edgecolor='#43A047', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + 1.75, y + 0.7, name, ha='center', va='center',
            fontsize=10, fontweight='bold')
    ax.text(x + 1.75, y + 0.15, info, ha='center', va='center',
            fontsize=8, color='gray')

# Methods under Branch 2
methods_no_cv = [
    (8, 3.5, 'pixelSplat\n(CVPR 2024)', '#BBDEFB', '10 FPS\nPSNR ~24.5'),
    (8, 1.2, 'Splatt3R\n(2025)', '#BBDEFB', 'Single image\nGenerative'),
]

for (x, y, name, color, info) in methods_no_cv:
    box = FancyBboxPatch((x, y), 3.5, 1.2,
                         boxstyle="round,pad=0.1",
                         facecolor=color, edgecolor='#1976D2', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + 1.75, y + 0.7, name, ha='center', va='center',
            fontsize=10, fontweight='bold')
    ax.text(x + 1.75, y + 0.15, info, ha='center', va='center',
            fontsize=8, color='gray')

# Arrows from branches to methods
ax.annotate('', xy=(2.25, 4.7), xytext=(3.25, 5.5),
            arrowprops=dict(arrowstyle='->', color='#43A047', lw=1.5))
ax.annotate('', xy=(2.25, 2.4), xytext=(3.25, 5.5),
            arrowprops=dict(arrowstyle='->', color='#43A047', lw=1.5))
ax.annotate('', xy=(9.75, 4.7), xytext=(10.25, 5.5),
            arrowprops=dict(arrowstyle='->', color='#1976D2', lw=1.5))
ax.annotate('', xy=(9.75, 2.4), xytext=(10.25, 5.5),
            arrowprops=dict(arrowstyle='->', color='#1976D2', lw=1.5))

# Phase 5 hint
hint_box = FancyBboxPatch((4.5, 0.2), 5, 0.8,
                          boxstyle="round,pad=0.1",
                          facecolor='#FFF3E0', edgecolor='#E65100', linewidth=1.5, linestyle='--')
ax.add_patch(hint_box)
ax.text(7, 0.6, 'VGGT / Foundation Models (Phase 5)', ha='center', va='center',
        fontsize=10, color='#E65100', style='italic')

plt.tight_layout()
plt.savefig('method_taxonomy.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. How Does It Work? (High-level)

Let's walk through the feed-forward pipeline step by step using MVSplat as the example.

### Step 1: Feature Extraction

Each input image is processed by a **shared U-Net encoder** to extract multi-scale features:

```
Image (H x W x 3)  ──►  U-Net Encoder  ──►  Feature Map (H/4 x W/4 x C)
```

### Step 2: Cross-view Geometry (Cost Volume)

The key geometric reasoning step:
1. Sample D depth hypotheses (planes) between d_min and d_max
2. For each depth, warp source features to reference view
3. Compute matching cost (feature variance)
4. Result: Cost Volume C(u, v, d) tells us how well each depth matches

```
Feature Maps + Camera Params  ──►  Plane Sweeping  ──►  Cost Volume (H x W x D x C)
```

### Step 3: Cost Volume Processing

A **3D U-Net** processes the Cost Volume to aggregate spatial and depth information.

### Step 4: Gaussian Prediction

For each pixel, predict Gaussian parameters:
- **Depth** → back-project to get 3D position
- **Covariance** → Gaussian shape (scale + rotation)
- **Opacity** → Gaussian transparency
- **Color** → directly from input image

### Step 5: Render with Standard 3DGS

The predicted Gaussians are rendered using the same rasterizer from Phase 1!

In [ ]:
# Visualize the MVSplat pipeline step by step

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

np.random.seed(42)

# Step 1: Input images
ax = axes[0, 0]
ax.set_title('Step 1: Input Images (2-5 views)', fontsize=11, fontweight='bold')
# Two "camera" views
for i, (cx, cy, label) in enumerate([(0.3, 0.5, 'View 1'), (0.7, 0.5, 'View 2')]):
    img = np.random.rand(8, 8, 3) * 0.3 + 0.4
    # Make a gradient pattern
    for r in range(8):
        for c in range(8):
            img[r, c, 0] = 0.3 + r/10
            img[r, c, 1] = 0.5 + c/16
            img[r, c, 2] = 0.4
    extent = [cx-0.18, cx+0.18, cy-0.18, cy+0.18]
    ax.imshow(img, extent=extent)
    ax.text(cx, cy-0.22, label, ha='center', fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# Step 2: Feature extraction
ax = axes[0, 1]
ax.set_title('Step 2: Feature Extraction', fontsize=11, fontweight='bold')
feat = np.random.rand(16, 16)
ax.imshow(feat, cmap='coolwarm', alpha=0.8)
ax.text(8, -1.5, 'Shared U-Net Encoder', ha='center', fontsize=9, style='italic')
ax.text(8, 17, 'Feature Map (H/4 x W/4 x C)', ha='center', fontsize=8, color='gray')
ax.axis('off')

# Step 3: Cost Volume
ax = axes[0, 2]
ax.set_title('Step 3: Cost Volume (Plane Sweeping)', fontsize=11, fontweight='bold')
# Show cost volume slices
n_slices = 5
for i in range(n_slices):
    offset = i * 0.15
    slice_data = np.random.rand(8, 8) * 0.5 + 0.25
    # Make one slice have a clear "minimum cost" pattern
    if i == 2:
        slice_data = np.ones((8, 8)) * 0.8
        slice_data[2:6, 2:6] = 0.2  # low cost = good match
    extent = [offset, offset + 0.6, offset, offset + 0.6]
    ax.imshow(slice_data, extent=extent, cmap='RdYlGn_r', 
             alpha=0.6, vmin=0, vmax=1)
    ax.text(offset + 0.65, offset + 0.3, f'd={i+1}', fontsize=7)

ax.set_xlim(-0.1, 1.4)
ax.set_ylim(-0.1, 1.2)
ax.text(0.6, -0.2, 'C(u,v,d): cost at each depth', ha='center', fontsize=8, color='gray')
ax.axis('off')

# Step 4: Gaussian prediction
ax = axes[1, 0]
ax.set_title('Step 4: Predict Gaussians', fontsize=11, fontweight='bold')
# Show depth map
x = np.linspace(-2, 2, 16)
y = np.linspace(-2, 2, 16)
X, Y = np.meshgrid(x, y)
depth = 2 + np.sin(X) * 0.5 + np.cos(Y) * 0.3
im = ax.imshow(depth, cmap='plasma')
ax.text(8, -1.5, 'Predicted depth + Gaussian params', ha='center', fontsize=9, style='italic')
plt.colorbar(im, ax=ax, label='Depth', fraction=0.046)
ax.axis('off')

# Step 5: 3D Gaussians
ax = axes[1, 1]
ax.set_title('Step 5: Pixel-aligned 3D Gaussians', fontsize=11, fontweight='bold')
n = 200
gx = np.random.randn(n) * 1.5
gy = np.random.randn(n) * 1.5
colors = plt.cm.viridis(np.random.rand(n))
for i in range(n):
    s = np.random.rand() * 0.15 + 0.05
    ellipse = Ellipse((gx[i], gy[i]), s, s * 0.8,
                     angle=np.random.rand()*180,
                     facecolor=colors[i], alpha=0.4, edgecolor='none')
    ax.add_patch(ellipse)
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_aspect('equal')
ax.text(0, -3.7, 'One Gaussian per pixel', ha='center', fontsize=9, style='italic')

# Step 6: Rendering
ax = axes[1, 2]
ax.set_title('Step 6: Novel View Rendering', fontsize=11, fontweight='bold')
# Simulated rendered image
rendered = np.zeros((16, 16, 3))
for r in range(16):
    for c in range(16):
        rendered[r, c, 0] = 0.3 + r/20
        rendered[r, c, 1] = 0.5 + c/25
        rendered[r, c, 2] = 0.35 + (r+c)/40
rendered = np.clip(rendered + np.random.randn(16, 16, 3) * 0.02, 0, 1)
ax.imshow(rendered)
ax.text(8, -1.5, 'Standard 3DGS rasterizer (from Phase 1)', ha='center', fontsize=9, style='italic')
ax.axis('off')

plt.tight_layout()
plt.savefig('feedforward_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

print("Complete pipeline: Images → Features → Cost Volume → Gaussians → Rendering")
print("The rendering step (Step 6) uses the same rasterizer we studied in Phase 1!")

## 6. Connection to SLAM & MVS

Feed-forward 3DGS has deep connections to concepts from SLAM (Phase 2) and DUSt3R (Phase 3).

### 6.1 Cost Volume in SLAM

The Cost Volume in MVSplat is the same concept used in SLAM dense mapping:

| Concept | In SLAM | In MVSplat |
|---------|---------|------------|
| Cost Volume | PatchMatch / MVSNet | Plane Sweeping |
| Purpose | Dense depth estimation | Gaussian parameter prediction |
| Processing | Regularization + winner-take-all | 3D U-Net + learned heads |
| Output | Depth map | Depth + Gaussian params |

### 6.2 Pixel-aligned Gaussians vs Surfels

| Concept | Surfel Mapping | Pixel-aligned Gaussians |
|---------|---------------|------------------------|
| Geometry | Oriented disk | 3D Gaussian ellipsoid |
| Placement | From depth at pixel | From predicted depth at pixel |
| Properties | Position, normal, radius, color | Position, covariance, opacity, color |
| Update | Fused over time | Predicted in one shot |

### 6.3 DUSt3R vs Feed-forward 3DGS

| Aspect | DUSt3R (Phase 3) | Feed-forward 3DGS (Phase 4) |
|--------|---------|------------------------|
| Output | Dense point cloud (Pointmap) | 3D Gaussians (renderable) |
| Rendering | Needs post-processing | Direct novel view synthesis |
| Architecture | ViT + Cross-attention | U-Net + Cost Volume (or attention) |
| Camera poses | Not needed (predicted) | Usually needed |
| Goal | 3D reconstruction | Novel view synthesis |

In [ ]:
# Visualize the relationship between phases

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 14)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_title('How Phase 4 Connects to Other Phases', fontsize=14, fontweight='bold')

# Phase boxes
phases = [
    (0.3, 4.0, 'Phase 1\n3DGS Foundation', '#FFCDD2', '#D32F2F'),
    (3.5, 4.0, 'Phase 2\n3DGS + SLAM', '#C8E6C9', '#2E7D32'),
    (6.7, 4.0, 'Phase 3\nDUSt3R', '#BBDEFB', '#1565C0'),
    (9.9, 4.0, 'Phase 4\nFeed-forward', '#FFF9C4', '#F57F17'),
]

for (x, y, text, color, edge_color) in phases:
    box = FancyBboxPatch((x, y), 2.8, 1.5,
                         boxstyle="round,pad=0.15",
                         facecolor=color, edgecolor=edge_color, linewidth=2)
    ax.add_patch(box)
    ax.text(x + 1.4, y + 0.75, text, ha='center', va='center',
            fontsize=10, fontweight='bold')

# Arrows showing connections
# Phase 1 -> Phase 4: Rendering, Gaussian model
ax.annotate('', xy=(10.5, 4.0), xytext=(1.7, 4.0),
            arrowprops=dict(arrowstyle='->', color='#D32F2F', lw=2,
                           connectionstyle='arc3,rad=-0.4'))
ax.text(6, 2.5, 'Gaussian model\n& rendering', ha='center', fontsize=9, color='#D32F2F')

# Phase 2 -> Phase 4: Cost Volume, MVS
ax.annotate('', xy=(10.5, 4.5), xytext=(6.3, 4.5),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2,
                           connectionstyle='arc3,rad=-0.2'))
ax.text(8.3, 3.5, 'Cost Volume\n& MVS', ha='center', fontsize=9, color='#2E7D32')

# Phase 3 -> Phase 4: Dense prediction paradigm
ax.annotate('', xy=(10.2, 5.0), xytext=(9.5, 5.0),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))
ax.text(9.8, 5.9, 'Dense pixel-wise\nprediction', ha='center', fontsize=9, color='#1565C0')

# Shared concepts below
concepts = [
    (1.0, 1.0, 'Differentiable\nRendering', '#FFCDD2'),
    (4.0, 1.0, 'Camera\nGeometry', '#C8E6C9'),
    (7.0, 1.0, 'Cross-view\nMatching', '#BBDEFB'),
    (10.0, 1.0, 'Neural Gaussian\nPrediction', '#FFF9C4'),
]

for (x, y, text, color) in concepts:
    box = FancyBboxPatch((x, y), 2.5, 0.8,
                         boxstyle="round,pad=0.1",
                         facecolor=color, edgecolor='gray', linewidth=1, linestyle='--')
    ax.add_patch(box)
    ax.text(x + 1.25, y + 0.4, text, ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('phase_connections.png', dpi=150, bbox_inches='tight')
plt.show()

print("Phase 4 builds on concepts from all previous phases:")
print("  Phase 1: Gaussian model, rendering pipeline, differentiable rendering")
print("  Phase 2: Cost Volume, multi-view stereo, dense mapping")
print("  Phase 3: Dense pixel-wise prediction, cross-view attention")

## 7. A Simple Conceptual Demo

Let's build a toy example to demonstrate the core concept: **predicting Gaussians from image features**.

This is a highly simplified version that shows the key idea without the full architecture.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
class ToyGaussianPredictor(nn.Module):
    """
    A simplified feed-forward Gaussian predictor for educational purposes.
    
    Given a feature map (simulating the output of an encoder),
    predicts per-pixel Gaussian parameters:
    - Depth (scalar per pixel)
    - Scale (2D, for the projected Gaussian)
    - Opacity (scalar)
    
    This demonstrates the core concept without Cost Volume or real images.
    """
    
    def __init__(self, feature_dim=32):
        super().__init__()
        
        # Depth prediction head
        self.depth_head = nn.Sequential(
            nn.Conv2d(feature_dim, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 1, 1),
            nn.Softplus()  # Depth must be positive
        )
        
        # Scale prediction head (2D scale for simplicity)
        self.scale_head = nn.Sequential(
            nn.Conv2d(feature_dim, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 2, 1),
            nn.Softplus()  # Scale must be positive
        )
        
        # Opacity prediction head
        self.opacity_head = nn.Sequential(
            nn.Conv2d(feature_dim, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 1, 1),
            nn.Sigmoid()  # Opacity in [0, 1]
        )
    
    def forward(self, features):
        """
        Args:
            features: [B, C, H, W] feature map
        Returns:
            dict with depth, scale, opacity per pixel
        """
        depth = self.depth_head(features)      # [B, 1, H, W]
        scale = self.scale_head(features)      # [B, 2, H, W]
        opacity = self.opacity_head(features)  # [B, 1, H, W]
        
        return {
            'depth': depth,
            'scale': scale,
            'opacity': opacity,
        }


# Create model and run on synthetic features
model = ToyGaussianPredictor(feature_dim=32)

# Simulated feature map (as if from an encoder)
B, C, H, W = 1, 32, 16, 16
features = torch.randn(B, C, H, W)

# Forward pass
with torch.no_grad():
    output = model(features)

print("Toy Gaussian Predictor Output:")
print(f"  Input features: {features.shape}")
print(f"  Predicted depth: {output['depth'].shape} "
      f"(range: [{output['depth'].min():.3f}, {output['depth'].max():.3f}])")
print(f"  Predicted scale: {output['scale'].shape} "
      f"(range: [{output['scale'].min():.3f}, {output['scale'].max():.3f}])")
print(f"  Predicted opacity: {output['opacity'].shape} "
      f"(range: [{output['opacity'].min():.3f}, {output['opacity'].max():.3f}])")
print(f"\n  Total Gaussians predicted: {H * W} (one per pixel)")
print(f"  This is the core idea: a network predicts Gaussian params from features!")

In [ ]:
# Visualize the predicted parameters

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

depth = output['depth'][0, 0].numpy()
scale_x = output['scale'][0, 0].numpy()
scale_y = output['scale'][0, 1].numpy()
opacity = output['opacity'][0, 0].numpy()

# Depth map
ax = axes[0]
im = ax.imshow(depth, cmap='plasma')
ax.set_title('Predicted Depth', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.046)
ax.axis('off')

# Scale X
ax = axes[1]
im = ax.imshow(scale_x, cmap='YlOrRd')
ax.set_title('Predicted Scale X', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.046)
ax.axis('off')

# Scale Y
ax = axes[2]
im = ax.imshow(scale_y, cmap='YlOrRd')
ax.set_title('Predicted Scale Y', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.046)
ax.axis('off')

# Opacity
ax = axes[3]
im = ax.imshow(opacity, cmap='gray', vmin=0, vmax=1)
ax.set_title('Predicted Opacity', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.046)
ax.axis('off')

plt.suptitle('Per-pixel Gaussian Parameter Prediction (Toy Example)', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('predicted_params.png', dpi=150, bbox_inches='tight')
plt.show()

print("Each pixel has its own predicted Gaussian parameters.")
print("In a real model, these would come from a Cost Volume + learned features.")

In [ ]:
def unproject_pixels_to_3d(depth_map, fx=500, fy=500, cx=8, cy=8):
    """
    Back-project pixel coordinates to 3D using predicted depth.
    This is how pixel-aligned Gaussians get their 3D positions.
    
    Args:
        depth_map: [H, W] predicted depth
        fx, fy: focal lengths
        cx, cy: principal point
    
    Returns:
        points_3d: [H, W, 3] 3D coordinates
    """
    H, W = depth_map.shape
    
    # Create pixel coordinate grid
    u = np.arange(W)  # column indices
    v = np.arange(H)  # row indices
    u, v = np.meshgrid(u, v)
    
    # Back-project: X = (u - cx) * Z / fx, Y = (v - cy) * Z / fy
    Z = depth_map
    X = (u - cx) * Z / fx
    Y = (v - cy) * Z / fy
    
    points_3d = np.stack([X, Y, Z], axis=-1)
    return points_3d


# Create a synthetic depth map (smooth surface)
H, W = 16, 16
u_grid = np.linspace(-2, 2, W)
v_grid = np.linspace(-2, 2, H)
U, V = np.meshgrid(u_grid, v_grid)
synthetic_depth = 3.0 + 0.5 * np.sin(U) + 0.3 * np.cos(V)  # Smooth surface

# Back-project to 3D
points_3d = unproject_pixels_to_3d(synthetic_depth, fx=10, fy=10, cx=W/2, cy=H/2)

# Visualize
fig = plt.figure(figsize=(16, 5))

# Depth map
ax1 = fig.add_subplot(131)
im = ax1.imshow(synthetic_depth, cmap='plasma')
ax1.set_title('Predicted Depth Map', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax1, fraction=0.046)
ax1.axis('off')

# 3D point cloud (top view)
ax2 = fig.add_subplot(132)
X_flat = points_3d[:, :, 0].flatten()
Y_flat = points_3d[:, :, 1].flatten()
Z_flat = points_3d[:, :, 2].flatten()
scatter = ax2.scatter(X_flat, Z_flat, c=Z_flat, cmap='plasma', s=10, alpha=0.7)
ax2.set_xlabel('X')
ax2.set_ylabel('Z (depth)')
ax2.set_title('3D Points (Top View)', fontsize=11, fontweight='bold')
ax2.set_aspect('equal')
plt.colorbar(scatter, ax=ax2, fraction=0.046)

# 3D point cloud (side view)
ax3 = fig.add_subplot(133)
scatter = ax3.scatter(X_flat, Y_flat, c=Z_flat, cmap='plasma', s=10, alpha=0.7)
ax3.set_xlabel('X')
ax3.set_ylabel('Y')
ax3.set_title('3D Points (Front View)', fontsize=11, fontweight='bold')
ax3.set_aspect('equal')
plt.colorbar(scatter, ax=ax3, fraction=0.046)

plt.tight_layout()
plt.savefig('back_projection.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Back-projected {H*W} pixels to 3D using predicted depth.")
print(f"Each 3D point becomes the CENTER of a pixel-aligned Gaussian.")
print(f"The network also predicts scale, rotation, and opacity for each.")

## 8. Training the Feed-forward Model

How is the network trained? The key insight: **train on many scenes, evaluate on novel views**.

### Training Pipeline

```
For each training batch:
  1. Sample a scene from dataset (RE10K / ACID)
  2. Sample N input views + M target views
  3. Feed N input views to network → predict Gaussians
  4. Render predicted Gaussians from M target viewpoints
  5. Compare rendered images with ground truth (L1 + SSIM loss)
  6. Backprop through renderer AND network
```

### Key Difference from Phase 1

| Aspect | Phase 1 Training | Phase 4 Training |
|--------|-----------------|------------------|
| What's optimized | Gaussian parameters | Network weights |
| Gradients flow to | Positions, scales, rotations, SH | Conv weights, attention params |
| Data | Single scene, all views | Many scenes, sampled views |
| Loss | L1 + D-SSIM on known views | L1 + SSIM on held-out views |
| Density control | Clone / Split / Prune | Not needed (fixed per-pixel) |

In [ ]:
# Visualize the training loop

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 9)
ax.axis('off')
ax.set_title('Feed-forward 3DGS Training Loop', fontsize=14, fontweight='bold')

# Step boxes
steps = [
    # (x, y, w, h, text, color)
    (0.5, 7, 3.5, 1.2, 'Sample Scene\nfrom Dataset', '#E8F5E9'),
    (0.5, 5, 3.5, 1.2, 'Select N Input Views\n+ M Target Views', '#E3F2FD'),
    (5, 7, 3.5, 1.2, 'Network\nForward Pass', '#FFF9C4'),
    (5, 5, 3.5, 1.2, 'Predict\nGaussians', '#FFF9C4'),
    (9.5, 7, 4, 1.2, 'Render from\nTarget Viewpoints', '#F3E5F5'),
    (9.5, 5, 4, 1.2, 'Compute Loss\nL = L1 + SSIM', '#FFEBEE'),
    (5, 2.5, 3.5, 1.2, 'Backprop Through\nRenderer + Network', '#FFCCBC'),
    (5, 0.5, 3.5, 1.2, 'Update Network\nWeights (Adam)', '#E0F7FA'),
]

for (x, y, w, h, text, color) in steps:
    box = FancyBboxPatch((x, y), w, h,
                         boxstyle="round,pad=0.1",
                         facecolor=color, edgecolor='gray', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=9)

# Arrows
arrows = [
    ((2.25, 7), (5, 7.6), 'Input views'),
    ((2.25, 5), (5, 5.6), 'Target views'),
    ((6.75, 7), (6.75, 6.2), ''),
    ((8.5, 5.6), (9.5, 5.6), 'Render'),
    ((8.5, 7.6), (9.5, 7.6), 'Gaussians'),
    ((11.5, 7), (11.5, 6.2), ''),
    ((11.5, 5), (8.5, 3.5), 'Gradients'),
    ((6.75, 2.5), (6.75, 1.7), ''),
]

for (start, end, label) in arrows:
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle='->', color='#455A64', lw=1.5))
    if label:
        mid_x = (start[0] + end[0]) / 2
        mid_y = (start[1] + end[1]) / 2
        ax.text(mid_x, mid_y + 0.2, label, fontsize=7, color='gray', ha='center')

# Loop arrow back to top
ax.annotate('', xy=(0.8, 8.2), xytext=(5, 1.7),
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=2,
                           connectionstyle='arc3,rad=0.5', linestyle='--'))
ax.text(0.3, 4.5, 'Repeat for\nall scenes', fontsize=9, color='#E65100',
        ha='center', style='italic', rotation=90)

plt.tight_layout()
plt.savefig('training_loop.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key insight: the loss function is the SAME as Phase 1 (photometric loss),")
print("but gradients flow through the NETWORK, not the Gaussian parameters directly.")

## 9. Phase 4 Learning Path

### Notebook Structure

| # | Topic | Key Concepts |
|---|-------|-------------|
| 00 | Overview (this notebook) | Paradigm shift, method landscape |
| 01 | Cost Volume & Plane Sweeping | MVS geometry, depth hypotheses |
| 02 | Pixel-aligned Gaussians | Back-projection, structured layout |
| 03 | MVSplat Architecture | Complete architecture deep dive |
| 04 | pixelSplat & Implicit Geometry | No Cost Volume, cross-attention |
| 05 | Training & Loss Design | Dataset, losses, strategies |
| 06 | MVSplat Code Walkthrough | Official code analysis |
| 07 | Inference & Evaluation | Running & benchmarking |
| 08 | DepthSplat & 2025 Advances | Depth priors, new methods |
| 09 | MVSplat vs pixelSplat | Systematic comparison |

In [ ]:
# Summary of what we covered

summary = """
╔═══════════════════════════════════════════════════════════════════════╗
║                    Phase 4 Overview - Summary                        ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. PARADIGM SHIFT                                                   ║
║     - From per-scene optimization to pretrained feed-forward         ║
║     - 10000x speedup (30 min → <1 sec)                              ║
║     - Cross-scene generalization                                     ║
║                                                                      ║
║  2. KEY METHODS                                                      ║
║     - MVSplat: Cost Volume + Gaussian prediction (fast, robust)      ║
║     - pixelSplat: Implicit geometry via attention (elegant)          ║
║     - DepthSplat: Depth prior + Cost Volume (2025, best quality)    ║
║                                                                      ║
║  3. PIXEL-ALIGNED GAUSSIANS                                          ║
║     - One Gaussian per pixel (structured, regular)                   ║
║     - Center = back-projected pixel at predicted depth               ║
║     - No adaptive density control needed                             ║
║                                                                      ║
║  4. CONNECTIONS                                                      ║
║     - Phase 1: Same rendering pipeline                               ║
║     - Phase 2: Cost Volume from MVS                                  ║
║     - Phase 3: Dense pixel-wise prediction paradigm                  ║
║                                                                      ║
║  5. TRAINING                                                         ║
║     - Train on many scenes, evaluate on novel views                  ║
║     - Same photometric loss, gradients to network weights            ║
║                                                                      ║
╚═══════════════════════════════════════════════════════════════════════╝
"""
print(summary)

## 10. Getting Started

### Prerequisites

```bash
# MVSplat
git clone https://github.com/donydchen/mvsplat.git
cd mvsplat
conda create -n mvsplat python=3.10
conda activate mvsplat
pip install torch==2.1.0 torchvision --index-url https://download.pytorch.org/whl/cu121
pip install -r requirements.txt
pip install gsplat

# Download pretrained model
mkdir -p checkpoints
# Download from project page: mvsplat_re10k.ckpt
```

### Key Resources

- MVSplat paper: [arXiv:2403.14627](https://arxiv.org/abs/2403.14627)
- pixelSplat paper: [arXiv:2312.12337](https://arxiv.org/abs/2312.12337)
- DepthSplat paper: [arXiv:2412.18010](https://arxiv.org/abs/2412.18010)

---

## What's Next?

In the next notebook, we'll deep dive into the **Cost Volume and Plane Sweeping** mechanism:

**[01_cost_volume_plane_sweeping.ipynb](./01_cost_volume_plane_sweeping.ipynb)** - The geometric foundation of MVSplat

---

## References

1. MVSplat: https://donydchen.github.io/mvsplat/
2. pixelSplat: https://davidcharatan.com/pixelsplat/
3. DepthSplat: https://arxiv.org/abs/2412.18010
4. 3D Gaussian Splatting: https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/
5. gsplat: https://docs.gsplat.studio/